# Generating Shakespearean Text Using a Character RNN (GRU)

This notebook is based on Geron's [notebook](https://github.com/ageron/handson-ml3/blob/main/16_nlp_with_rnns_and_attention.ipynb).

Presented here in accordance with Apache 2.0 license.

***

<font color='green'>Hello Profs, thank you for suggesting this notebook to use for our project, you can find all of our markdowns and explorations marked up in this **green** color.</font>

***

In [1]:
import tensorflow as tf

***

<font color='green'> As we took in class, this note book generates Shakespearean Text. We are going to experiment with different parameters and prompts. </font>

***

## Creating the Training Dataset

Let's download the Shakespeare data from Andrej Karpathy's [char-rnn project](https://github.com/karpathy/char-rnn/)

In [2]:
shakespeare_url = "https://homl.info/shakespeare"  # shortcut URL
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

## Explore the Dataset

In [3]:
print(type(shakespeare_text))

<class 'str'>


In [4]:
print(f'Number of characters in the text: {len(shakespeare_text):,}')

Number of characters in the text: 1,115,394


In [5]:
# shows the first 80 characters in the text:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


***

<font color='green'> Let's print the last 102 characters in the text: </font>

***

In [45]:
# shows the last 102 characters in the text:
print(shakespeare_text[-102:])

ANTONIO:
Noble Sebastian,
Thou let'st thy fortune sleep--die, rather; wink'st
Whiles thou art waking.



In [6]:
# shows all distinct characters (after converting to lower case)
distinct_chars = "".join(sorted(set(shakespeare_text.lower())))
print(f'Number of distinct characters: {len(distinct_chars)}')

Number of distinct characters: 39


In [7]:
distinct_chars

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

***

<font color='green'> We found it weird the number '3' and a '$' is being used in shakespearean text, let's look at instances where they're used: </font>

***

In [49]:
index = shakespeare_text.find("$")
if index != -1:
    print(shakespeare_text[index-15:index+15])
else:
    print("Character not found")

ing; my sea sha$l suck them dr


In [50]:
index = shakespeare_text.find("$")
if index != -1:
    print(shakespeare_text[index-125:index+125])
else:
    print("Character not found")

d once again proclaim us King of England.
You are the fount that makes small brooks to flow:
Now stops thy spring; my sea sha$l suck them dry,
And swell so much the higher by their ebb.
Hence with him to the Tower; let him not speak.
And, lords, towa


***

<font color='green'> Maybe this dataset was first collected using OCR or maybe this is just a typo so the '$' here should have been an 'L'. So the final word here would be 'shall'. Let's see how many times it's being repeated: </font>

***

In [51]:
char = "$"
count = shakespeare_text.count(char)
print("The character '{}' appears {} times in the entire text.".format(char, count))

The character '$' appears 1 times in the entire text.


***

<font color='green'>What about the number '3' ? </font>

***

In [53]:
index = shakespeare_text.find("3")
if index != -1:
    print(shakespeare_text[index-15:index+15])
else:
    print("Character not found")

cile them all.
3 KING HENRY VI


In [67]:
index = shakespeare_text.find("3")
if index != -1:
    print(shakespeare_text[index-90:index+80])
else:
    print("Character not found")

ome, cousin you shall be the messenger.

EXETER:
And I, I hope, shall reconcile them all.
3 KING HENRY VI

RICHARD:
Brother, though I be youngest, give me leave.

EDWARD:


***

<font color='green'> Let's see how many times it's being repeated: </font>

***

In [68]:
char = "3"
count = shakespeare_text.count(char)
print("The character '{}' appears {} times in the entire text.".format(char, count))

The character '3' appears 27 times in the entire text.


***

<font color='green'> This could be a typo as well, one must compare with the orginal text to identify what the '3' here signifies. </font>

***

## Preprocess the Data

For the students: 

What do the commands below do, specifically `TextVectorization` and its `adapt` method? 

Looking at the [reference](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization) for TextVectorization or in Geron p.471 will be useful.

***
<font color='green'>From the reference above this was the definitions given:</font>

**TextVectorization:** A preprocessing layer which maps text features to integer sequences.

<font color='green'> **Why  we use TextVectorization?** As we saw above our input is of string type, consisting of characters, our NN model cannot process characters, that's why we need to convert it to some form of number sets that represent the input string. </font>

***

In [8]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character",
                                                   standardize="lower")

***

<font color='green'> split = "character" will split by each character. The other option would be splitting by "whitespace" which will split each word. There's also the option of not splitting by calling None. </font>

<font color='green'> standardize="lower" will apply lower case to input text. Other options include "strip_punctuation" to remove punctuation and "lower_and_strip_punctuation" to remove both and there's also None.</font>

***

In [9]:
text_vec_layer

***

<font color='green'> Applying our adapt method to the text_vec_layer: </font>

***

In [10]:
text_vec_layer.adapt([shakespeare_text])

***
**adapt method:** Computes a vocabulary of string terms from tokens in a dataset.

The adapt() method is used to train a TextVectorization layer in TensorFlow. It builds the vocabulary of all string tokens in the input dataset and computes their occurrence count and document frequency (if output_mode is set to 'tf-idf'). The layer keeps the vocabulary static with respect to any compiled TensorFlow graphs, so if the layer is adapted a second time, any models using the layer will need to be re-compiled. The adapt() method can be passed either a tf.data.Dataset or a numpy array as input, and the batch size and number of steps can be specified as arguments. Note that if you pass a tf.data.Dataset as input, the batch size should not be specified, as the batch size is generated by the dataset.

<font color='green'> Okay so we want text to be converted to a series of numbers, but we also we want the same words to be represented by the same set of numbers every time that it's used in the text. So we can see here, that's what our adapt method is looking for: the vocabulary that our text has. This is very useful. </font>

***

In [11]:
text_vec_layer([shakespeare_text])

<tf.Tensor: shape=(1, 1115394), dtype=int64, numpy=array([[21,  7, 10, ..., 22, 28, 12]], dtype=int64)>

<font color='green'> shape=(1, 1115394) is the total number of charaters in our dataset</font>

In [12]:
encoded = text_vec_layer([shakespeare_text])[0]

* We set split="character" to get character-level encoding rather than the default word-level encoding, 
* We use standardize="lower" to convert the text to lowercase (which will simplify the task)

Next:
* Each character is now mapped to an integer, starting at 2. The `TextVectorization`
layer reserved the value 0 for padding tokens, and it reserved 1 for unknown characters. We won’t need either of these tokens for now, so let’s subtract 2 from the
character IDs

In [13]:
encoded

<tf.Tensor: shape=(1115394,), dtype=int64, numpy=array([21,  7, 10, ..., 22, 28, 12], dtype=int64)>

In [14]:
encoded -= 2  # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2  # number of distinct chars = 39
dataset_size = len(encoded)  # total number of chars = 1,115,394

In [15]:
encoded

<tf.Tensor: shape=(1115394,), dtype=int64, numpy=array([19,  5,  8, ..., 20, 26, 10], dtype=int64)>

In [16]:
print(f'Number of distinct tokens: {n_tokens}')

Number of distinct tokens: 39


In [17]:
print(f'Number of tokens: {dataset_size:,}')

Number of tokens: 1,115,394


we can turn this very long sequence into a dataset of windows that we can then use to train a sequence-to-sequence RNN. The targets will be similar to the inputs, but shifted by one time step into the “future”. For example, one sample in the dataset may be a sequence of character IDs representing the text “to be or not to b” (without the final “e”), and the corresponding target — a sequence of character IDs representing the text “o be or not to be” (with the final “e”, but without the leading “t”). Let’s write a small utility function to convert a long sequence of character IDs into a dataset of input/target window pairs:

In [18]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [19]:
# a simple example using to_dataset()
# There's just one sample in this dataset: the input represents "to b" and the
# output represents "o be"
list(to_dataset(text_vec_layer(["To be"])[0], length=4))

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


[(<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 4,  5,  2, 23]], dtype=int64)>,
  <tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 5,  2, 23,  3]], dtype=int64)>)]

In [20]:
length = 100
tf.random.set_seed(42)
train_set = to_dataset(encoded[:1_000_000], length=length, shuffle=True,
                       seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length=length)
test_set = to_dataset(encoded[1_060_000:], length=length)

In [21]:
train_set

<PrefetchDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

## Building and Training the Char-RNN Model

**Warning**: the following code may one or two hours to run, depending on your GPU. Without a GPU, it may take over 24 hours. If you don't want to wait, just skip the next two code cells and run the code below to download a pretrained model.

***

<font color='green'> We tried to run the training of the model, but it took more than 3 hours and still need 2 more hours to complete all 10 epochs. So we decided to go with the pre-trained model. </font>

***

**Note**: the `GRU` class will only use cuDNN acceleration (assuming you have a GPU) when using the default values for the following arguments: `activation`, `recurrent_activation`, `recurrent_dropout`, `unroll`, `use_bias` and `reset_after`.

We use an Embedding layer as the first layer, to encode the character IDs. The Embedding layer’s number of input dimensions is the number of distinct character IDs, and the number of output dimensions is a hyperparameter you can tune—we’ll set it to 16 for now. Whereas the inputs of the Embedding layer will be 2D tensors of shape [batch size, window length], the output of the Embedding layer will be a 3D tensor of shape [batch size, window length, embedding size].

See Geron p.466-469 for a great review about the Embedding layer.
See Geron p.571-572 (or [Wikipedia](https://en.wikipedia.org/wiki/Gated_recurrent_unit)) for a review about Gated Recurrect Unit (GRU).

In [22]:
'''
tf.random.set_seed(42)  # ensures reproducibility on CPU
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "my_shakespeare_model", monitor="val_accuracy", save_best_only=True)
history = model.fit(train_set, validation_data=valid_set, epochs=10,
                    callbacks=[model_ckpt])
'''

'\ntf.random.set_seed(42)  # ensures reproducibility on CPU\nmodel = tf.keras.Sequential([\n    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),\n    tf.keras.layers.GRU(128, return_sequences=True),\n    tf.keras.layers.Dense(n_tokens, activation="softmax")\n])\nmodel.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",\n              metrics=["accuracy"])\nmodel_ckpt = tf.keras.callbacks.ModelCheckpoint(\n    "my_shakespeare_model", monitor="val_accuracy", save_best_only=True)\nhistory = model.fit(train_set, validation_data=valid_set, epochs=10,\n                    callbacks=[model_ckpt])\n'

In [23]:
'''
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),  # no <PAD> or <UNK> tokens
    model
])
'''

'\nshakespeare_model = tf.keras.Sequential([\n    text_vec_layer,\n    tf.keras.layers.Lambda(lambda X: X - 2),  # no <PAD> or <UNK> tokens\n    model\n])\n'

***

<font color='green'> Unfortunately we cannot do hyperparameter tuning on a pre-trained model, unless we copy this model and and unfreeze the last layer. </font>

***

If you don't want to wait for training to complete, Geron pretrained a model for us. The following code will download it. Uncomment the last line if you want to use it instead of the model trained above.

In [24]:
from pathlib import Path

# extra code – downloads a pretrained model
url = "https://github.com/ageron/data/raw/main/shakespeare_model.tgz"
path = tf.keras.utils.get_file("shakespeare_model.tgz", url, extract=True)
model_path = Path(path).with_name("shakespeare_model")
shakespeare_model = tf.keras.models.load_model(model_path)

352865/352865 [==============================] - 0s 1us/step


In [25]:
y_proba = shakespeare_model.predict(["To be or not to b"])[0, -1]
y_pred = tf.argmax(y_proba)  # choose the most probable character ID
text_vec_layer.get_vocabulary()[y_pred + 2]

1/1 [==============================] - 0s 453ms/step


'e'

## Generating Fake Shakespearean Text

To generate new text using the char-RNN model, we could feed it some text, make
the model predict the most likely next letter, add it to the end of the text, then give
the extended text to the model to guess the next letter, and so on. This is called greedy
decoding. But in practice this often leads to the same words being repeated over and over again. Instead, we can sample the next character randomly, with a probability
equal to the estimated probability, using TensorFlow’s tf.random.categorical()
function. This will generate more diverse and interesting text. The categorical()
function samples random class indices, given the class log probabilities (logits). 

Example:

In [26]:
log_probas = tf.math.log([[0.5, 0.4, 0.1]])  # probas = 50%, 40%, and 10%
tf.random.set_seed(42)
tf.random.categorical(log_probas, num_samples=8)  # draw 8 samples

<tf.Tensor: shape=(1, 8), dtype=int64, numpy=array([[0, 1, 0, 2, 1, 0, 0, 1]], dtype=int64)>

To have more control over the diversity of the generated text, we can divide the logits
by a number called the temperature, which we can tweak as we wish. A temperature
close to zero favors high-probability characters, while a high temperature gives all
characters an equal probability. Lower temperatures are typically preferred when
generating fairly rigid and precise text, such as mathematical equations, while higher
temperatures are preferred when generating more diverse and creative text. 

In [27]:
def next_char(text, temperature=1):
    y_proba = shakespeare_model.predict([text])[0, -1:]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id + 2]

In [28]:
def extend_text(text, n_chars=50, temperature=1):
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text

In [29]:
tf.random.set_seed(42)  # ensures reproducibility on CPU

In [30]:
print(extend_text("To be or not to be", temperature=0.01))

1/1 [==============================] - 0s 44ms/step
To be or not to be the duke
as it is a proper strange death,
and the


In [31]:
print(extend_text("To be or not to be", temperature=1))

1/1 [==============================] - 0s 44ms/step
To be or not to behold?

second push:
gremio, lord all, a sistermen,


In [32]:
print(extend_text("To be or not to be", temperature=100))

1/1 [==============================] - 0s 45ms/step
To be or not to bef ,mt'&o3fpadm!$
wh!nse?bws3est--vgerdjw?c-y-ewznq
